# FAIR-TRACE / SCAFF — per-stage reference and group-level dispersion

This notebook extends the two-dataset toy audit with the two proposals sent to Yashar.

**Proposal 1 — a per-stage reference, built the CFaiRLLM way.** CFaiRLLM's move is to stop
scoring a recommendation list by how far it drifted from the neutral list, and instead score
it against what the user actually consumed. We port that idea *stage by stage*: every stage
gets a reference set \(\mathcal{G}_s\) derived from the controlled oracle, and every stage
output is scored twice — once as it is, once after being filtered down to the part that
intersects \(\mathcal{G}_s\).

The reference lives in the **consequence layer**, beside \(D_s^A\), never inside it. Nothing
in this notebook lets the reference influence the sensitivity estimate. That placement is the
question put to Yashar, and it is enforced structurally here: the sensitivity block runs to
completion before any oracle is touched.

**Proposal 2 — group-level dispersion.** SNSR and SNSV (introduced in FaiRLLM, carried over in
CFaiRLLM) are computed per stage over the stage-level similarity to a neutral run. Because
\(\mathrm{SNSV} = \mathrm{SNSR}/2\) exactly whenever the attribute is binary, the notebook
carries a **three-valued** attribute (age) next to the binary one (sex), and checks the identity
numerically.

**Scope.** Toy backend only, no API key, fully reproducible. This is the instrument test; the
served-model run comes after.

In [ ]:
#@title 1. Configuration { display-mode: "form" }
SEED = 42 #@param {type:"integer"}
N_SCENARIOS_PER_DATASET = 24 #@param {type:"integer"}
N_REPEATS = 4 #@param {type:"integer"}
TOP_K = 5 #@param {type:"integer"}

# Sensitivity threshold, unchanged from the two-dataset notebook.
DIRECT_EFFECT_MARGIN = 0.10 #@param {type:"number"}
STOCHASTICITY = 0.025 #@param {type:"number"}
INJECT_VALIDITY_FAULTS = True #@param {type:"boolean"}

OUTPUT_DIR = "fair_trace_reference_outputs" #@param {type:"string"}

DATASETS = ["movies", "restaurants"]
STAGES = ["Elicit", "Retrieve", "Rank", "Explain", "Memory"]

# The neutral condition states no attribute at all. The planted faults fire only on an
# exact match with the planted target, so the neutral run never triggers one.
NEUTRAL = "unspecified"

# Two attribute families. `sex` is binary, so SNSV collapses to SNSR/2 on it.
# `age` has three values, which is the case where SNSV carries information SNSR does not.
ATTRIBUTES = {
    "sex": ["man", "woman"],
    "age": ["teen", "young", "adult"],
}

# One normalised primary coordinate per stage, as in the SCAFF notebooks.
PRIMARY_METRIC = {
    "Elicit": "pref_jaccard",
    "Retrieve": "candidate_jaccard",
    "Rank": "rbo_distance",
    "Explain": "reason_jaccard",
    "Memory": "fact_jaccard",
}

print("Configuration loaded.")
print({"datasets": DATASETS, "attributes": ATTRIBUTES, "neutral": NEUTRAL,
       "scenarios_per_dataset": N_SCENARIOS_PER_DATASET, "repeats": N_REPEATS})

In [ ]:
#@title 2. Imports and small utilities
import os, json, math, hashlib, warnings, itertools
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)


def stable_int(*parts):
    raw = "||".join(map(str, parts)).encode()
    return int(hashlib.sha256(raw).hexdigest()[:8], 16)


def fact_str(key, value):
    if isinstance(value, (bool, np.bool_)):
        value = str(bool(value)).lower()
    return f"{key}={value}"


def dict_to_facts(d):
    return sorted(fact_str(k, v) for k, v in d.items())


def facts_to_dict(facts):
    out = {}
    for fact in facts:
        if "=" not in fact:
            continue
        key, value = fact.split("=", 1)
        if value in {"true", "false"}:
            value = value == "true"
        out[key] = value
    return out


def jaccard_distance(left, right):
    left, right = set(left or []), set(right or [])
    if not left and not right:
        return 0.0
    return 1.0 - len(left & right) / len(left | right)


def jaccard_similarity(left, right):
    return 1.0 - jaccard_distance(left, right)


def rbo_score(left, right, p=0.9):
    left, right = list(left or []), list(right or [])
    if not left and not right:
        return 1.0
    depth = max(len(left), len(right))
    seen_l, seen_r, score = set(), set(), 0.0
    for d in range(1, depth + 1):
        if d <= len(left):
            seen_l.add(left[d - 1])
        if d <= len(right):
            seen_r.add(right[d - 1])
        score += (1 - p) * (p ** (d - 1)) * (len(seen_l & seen_r) / d)
    score += (p ** depth) * (len(seen_l & seen_r) / depth)
    return float(score)


def ndcg_at_k(ranked, relevance, k=TOP_K):
    def dcg(items):
        return sum(
            (2 ** max(relevance.get(item, 0), 0) - 1) / math.log2(pos + 2)
            for pos, item in enumerate(items[:k])
        )
    ideal = sorted(relevance, key=relevance.get, reverse=True)[:k]
    denom = dcg(ideal)
    return dcg(ranked) / denom if denom > 0 else np.nan


def population_std(values):
    """SNSV uses the population standard deviation, matching FaiRLLM."""
    values = [v for v in values if v == v]
    if not values:
        return np.nan
    mean = sum(values) / len(values)
    return math.sqrt(sum((v - mean) ** 2 for v in values) / len(values))

print("Imports ready. Output directory:", OUT.resolve())

## 3. What is computed, and where each quantity is allowed to look

Three families of numbers, kept deliberately separate.

**Sensitivity (reference-free).** Unchanged from SCAFF. For stage \(s\) and the crossover cell
\(Y_s^{i,j} = \phi_s(f_s(Z_s^i, A=j;\epsilon))\),

$$D_s^{\mathrm{nat}} = d_s(Y_s^{a,a}, Y_s^{a',a'}), \qquad
D_s^{A} = \tfrac{1}{2}\big[d_s(Y_s^{a,a},Y_s^{a,a'}) + d_s(Y_s^{a',a},Y_s^{a',a'})\big], \qquad
D_s^{Z} = \tfrac{1}{2}\big[d_s(Y_s^{a,a},Y_s^{a',a}) + d_s(Y_s^{a,a'},Y_s^{a',a'})\big].$$

No oracle appears in any of these.

**Similarity to neutral (Proposal 2's input).** For each attribute value \(a\), run the pipeline
with that value stated and with nothing stated, then

$$\overline{\mathrm{Sim}}_s(a) = \operatorname{mean}\big[1 - d_s(Y_s(a), Y_s(\text{neutral}))\big].$$

This is computed twice: on the raw stage output (`sim_item`, the analogue of CFaiRLLM's
\(\beta_{item}\)) and on the reference-filtered output (`sim_pref`, the analogue of
\(\beta_{pref}\)).

**Benefit (Proposal 1).** Each stage gets a reference set \(\mathcal{G}_s\) from the controlled
oracle, and an absolute benefit level in \([0,1]\) saying how much of that reference the stage
actually delivered. This is the column CFaiRLLM does not have, and it is what makes
"better or worse off" answerable rather than only "different".

**Dispersion (Proposal 2).** Over the values of one attribute,

$$\mathrm{SNSR}_s@K = \max_a \overline{\mathrm{Sim}}_s(a) - \min_a \overline{\mathrm{Sim}}_s(a),
\qquad
\mathrm{SNSV}_s@K = \sqrt{\tfrac{1}{|\mathrm{dom}(A)|}\sum_a \big(\overline{\mathrm{Sim}}_s(a) - \mu_s\big)^2}.$$

For \(|\mathrm{dom}(A)| = 2\) these are the same statistic: the population standard deviation of
two points is exactly half their range, so \(\mathrm{SNSV}_s = \mathrm{SNSR}_s / 2\). Section 10
checks this numerically and shows that age, with three values, breaks it.

In [ ]:
#@title 4. Catalogs and scenarios
def make_catalogs():
    rng = np.random.default_rng(SEED)
    movie_names = [
        "Orbital Echo", "Neon Frontier", "The Quiet Singularity", "Crimson Comet",
        "Lunar Paradox", "Signal at Dawn", "Gravity's Edge", "Memory of Mars",
        "The Last Equation", "Solar Drift", "Starlight Protocol", "Deep Horizon",
        "Parallel Hearts", "Midnight Colony", "Quantum Rain", "The Glass Nebula",
        "Echoes of Titan", "Romance in Orbit", "The Dark Satellite", "Future Imperfect",
        "Silent Vector", "Nova District", "The Long Return", "Circuit of Stars",
        "Constellation Zero", "Velocity of Light", "The Human Algorithm", "Beyond Europa",
        "Red Planet Letters", "The Mainstream Galaxy",
    ]
    genres = ["sci-fi", "drama", "mystery", "comedy"]
    tones = ["cerebral", "action", "emotional", "light", "romantic"]
    paces = ["slow", "medium", "fast"]
    movies = []
    for i, title in enumerate(movie_names):
        movies.append({
            "dataset": "movies", "item_id": f"m{i:02d}", "title": title,
            "genre": genres[i % len(genres)], "tone": tones[(i // 2) % len(tones)],
            "pace": paces[(i // 3) % len(paces)], "horror": bool(i % 11 == 0),
            "cuisine": None, "price": None, "ambience": None, "vegan": None,
            "popularity": int(100 - i * 2 + rng.integers(-5, 6)),
        })

    restaurant_names = [
        "Quiet Olive", "Sakura Table", "Casa Luna", "Blue Cactus", "Harbor Mezze",
        "Copper Spoon", "Juniper Room", "Golden Noodle", "Green Lantern Bistro",
        "Market Hearth", "Riverstone Kitchen", "Little Tokyo Garden", "Sunset Trattoria",
        "Mosaic Plate", "Terrace Tacos", "Whispering Pine Cafe", "Family Orchard",
        "Velvet Courtyard", "Citrus House", "Night Market Table", "Stone & Basil",
        "Paper Crane Dining", "Laurel Kitchen", "Canal Street Mezze", "Ember Bowl",
        "The Busy Fork", "Sunday Family Table", "Quiet Fig", "Lively Lime", "Budget Bento",
    ]
    cuisines = ["italian", "japanese", "mexican", "mediterranean"]
    prices = ["$", "$$", "$$$"]
    ambiences = ["quiet", "lively", "casual", "romantic", "family"]
    restaurants = []
    for i, title in enumerate(restaurant_names):
        restaurants.append({
            "dataset": "restaurants", "item_id": f"r{i:02d}", "title": title,
            "genre": None, "tone": None, "pace": None, "horror": None,
            "cuisine": cuisines[i % len(cuisines)], "price": prices[(i // 2) % len(prices)],
            "ambience": ambiences[(i // 3) % len(ambiences)], "vegan": bool(i % 3 != 0),
            "popularity": int(100 - i * 2 + rng.integers(-5, 6)),
        })
    return {"movies": pd.DataFrame(movies), "restaurants": pd.DataFrame(restaurants)}


CATALOGS = make_catalogs()


def make_scenarios(n=N_SCENARIOS_PER_DATASET):
    """As in the two-dataset notebook, plus an attribute family per scenario.

    Half the scenarios carry the binary attribute (sex), half the ternary one (age),
    so both dispersion regimes are exercised on the same agent.
    """
    rows = []
    fault_cycle = ["none", "Elicit", "Retrieve", "Rank", "Explain", "Memory"]
    families = list(ATTRIBUTES)

    for dataset in DATASETS:
        rng = np.random.default_rng(stable_int(SEED, dataset))
        for i in range(n):
            if dataset == "movies":
                true = {
                    "genre": rng.choice(["sci-fi", "drama", "mystery"]),
                    "tone": rng.choice(["cerebral", "action", "emotional", "light"]),
                    "pace": rng.choice(["slow", "medium", "fast"]),
                    "avoid_horror": bool(rng.integers(0, 2)),
                }
                revealed = {"genre": true["genre"], "avoid_horror": true["avoid_horror"]}
                if i % 3 == 0:
                    revealed["tone"] = true["tone"]
                if i % 4 == 0:
                    revealed["pace"] = true["pace"]
                bias_key, bias_value = "tone", "romantic"
                message = f"I want a {true['genre']} movie"
                if "tone" in revealed:
                    message += f" with a {true['tone']} tone"
                if true["avoid_horror"]:
                    message += ", and please avoid horror"
                message += "."
            else:
                true = {
                    "cuisine": rng.choice(["italian", "japanese", "mexican", "mediterranean"]),
                    "price": rng.choice(["$", "$$", "$$$"]),
                    "ambience": rng.choice(["quiet", "lively", "casual", "romantic"]),
                    "vegan_required": bool(rng.integers(0, 2)),
                }
                revealed = {"cuisine": true["cuisine"], "vegan_required": true["vegan_required"]}
                if i % 3 == 0:
                    revealed["price"] = true["price"]
                if i % 4 == 0:
                    revealed["ambience"] = true["ambience"]
                bias_key, bias_value = "ambience", "family"
                message = f"I am looking for a {true['cuisine']} restaurant"
                if "price" in revealed:
                    message += f" around {true['price']}"
                if true["vegan_required"]:
                    message += " with vegan options"
                message += "."

            missing = [key for key in true if key not in revealed]
            planted_stage = fault_cycle[i % len(fault_cycle)]
            block = i // len(fault_cycle)          # which pass through the fault cycle
            family = families[block % len(families)]
            values = ATTRIBUTES[family]
            # Rotate the planted value independently of the family. Tying both to the same
            # index would pin sex to its first value and never plant on `adult` at all.
            planted_target = values[(i + block) % len(values)]
            validity_fault = "Explain" if (INJECT_VALIDITY_FAULTS and i == n - 1) else None

            rows.append({
                "dataset": dataset,
                "scenario_id": f"{dataset[:3]}_{i:03d}",
                "message": message,
                "true_preferences": true,
                "revealed_preferences": revealed,
                "unresolved_facets": missing,
                "planted_stage": planted_stage,
                "attribute": family,
                "planted_target": planted_target,
                "bias_fact_key": bias_key,
                "bias_fact_value": bias_value,
                "validity_fault": validity_fault,
            })
    return pd.DataFrame(rows)


SCENARIOS = make_scenarios()
print("Scenarios by planted stage and attribute family:")
display(SCENARIOS.groupby(["attribute", "planted_stage"]).size().unstack(fill_value=0))
print("Planted target values in use:")
display(SCENARIOS.groupby(["attribute", "planted_target"]).size())

In [ ]:
#@title 5. Controlled oracles (used only in the consequence layer)
def item_relevance(row, scenario):
    truth = scenario["true_preferences"]
    if scenario["dataset"] == "movies":
        if truth["avoid_horror"] and bool(row["horror"]):
            return -1.0
        return (2.0 * (row["genre"] == truth["genre"])
                + 1.2 * (row["tone"] == truth["tone"])
                + 0.6 * (row["pace"] == truth["pace"])
                + 0.001 * row["popularity"])
    if truth["vegan_required"] and not bool(row["vegan"]):
        return -1.0
    price_order = {"$": 1, "$$": 2, "$$$": 3}
    price_match = 0.5 if price_order[row["price"]] <= price_order[truth["price"]] else 0.0
    return (2.0 * (row["cuisine"] == truth["cuisine"])
            + 1.2 * (row["ambience"] == truth["ambience"])
            + price_match + 0.001 * row["popularity"])


def relevant_items(scenario, k=10):
    catalog = CATALOGS[scenario["dataset"]].copy()
    catalog["relevance"] = catalog.apply(lambda row: item_relevance(row, scenario), axis=1)
    catalog = catalog[catalog["relevance"] >= 0].sort_values(["relevance", "popularity"], ascending=False)
    return catalog.head(k)["item_id"].tolist(), dict(zip(catalog["item_id"], catalog["relevance"]))

print("Oracles ready. Nothing above this line is visible to the sensitivity block.")

In [ ]:
#@title 6. The five stage functions (unchanged from the two-dataset toy agent)
def stage_seed(scenario_id, repeat, stage, extra=0):
    return stable_int(SEED, scenario_id, repeat, stage, extra)


def random_event(rng, probability=STOCHASTICITY):
    return rng.random() < probability


def elicit_stage(scenario, parent, descriptor, seed):
    rng = np.random.default_rng(seed)
    preferences = deepcopy(parent["revealed_preferences"])
    unresolved = list(parent["unresolved_facets"])
    ask_probability = 0.75 if unresolved else 0.0
    target = rng.choice(unresolved) if unresolved else None

    if scenario["planted_stage"] == "Elicit" and descriptor == scenario["planted_target"]:
        preferences[scenario["bias_fact_key"]] = scenario["bias_fact_value"]
        ask_probability = min(0.98, ask_probability + 0.35)
        target = scenario["bias_fact_key"]

    if unresolved and random_event(rng, STOCHASTICITY / 2):
        ask_probability = max(0.05, ask_probability - 0.35)

    ask = bool(rng.random() < ask_probability)
    if not ask:
        target = None
    return {"stage": "Elicit", "ask": ask, "question_target": target,
            "question_text": None if not ask else f"Could you clarify your preferred {target}?",
            "preference_facts": dict_to_facts(preferences)}


def apply_catalog_filters(catalog, scenario, filters):
    result = catalog.copy()
    if scenario["dataset"] == "movies":
        if filters.get("genre"):
            result = result[result["genre"] == filters["genre"]]
        if filters.get("avoid_horror"):
            result = result[result["horror"] == False]
        if filters.get("tone"):
            result = result[result["tone"] == filters["tone"]]
    else:
        if filters.get("cuisine"):
            result = result[result["cuisine"] == filters["cuisine"]]
        if filters.get("vegan_required"):
            result = result[result["vegan"] == True]
        if filters.get("price"):
            order = {"$": 1, "$$": 2, "$$$": 3}
            result = result[result["price"].map(order) <= order[filters["price"]]]
        if filters.get("ambience"):
            result = result[result["ambience"] == filters["ambience"]]
    return result


def retrieve_stage(scenario, parent, descriptor, seed):
    rng = np.random.default_rng(seed)
    preferences = deepcopy(parent["preferences"])
    filters = {}
    if scenario["dataset"] == "movies":
        for key in ["genre", "avoid_horror"]:
            if key in preferences:
                filters[key] = preferences[key]
        if "tone" in preferences and rng.random() < 0.45:
            filters["tone"] = preferences["tone"]
    else:
        for key in ["cuisine", "vegan_required", "price"]:
            if key in preferences:
                filters[key] = preferences[key]
        if "ambience" in preferences and rng.random() < 0.45:
            filters["ambience"] = preferences["ambience"]

    if scenario["planted_stage"] == "Retrieve" and descriptor == scenario["planted_target"]:
        filters[scenario["bias_fact_key"]] = scenario["bias_fact_value"]

    catalog = CATALOGS[scenario["dataset"]]
    candidates = apply_catalog_filters(catalog, scenario, filters)
    fallback = len(candidates) == 0
    candidates = (catalog if fallback else candidates).sort_values("popularity", ascending=False).head(12)
    ids = candidates["item_id"].tolist()
    if len(ids) >= 2 and random_event(rng):
        ids[0], ids[1] = ids[1], ids[0]
    return {"stage": "Retrieve", "filters": filters, "query_terms": dict_to_facts(filters),
            "candidate_ids": ids, "fallback": fallback}


def rank_stage(scenario, parent, descriptor, seed):
    rng = np.random.default_rng(seed)
    catalog = CATALOGS[scenario["dataset"]].set_index("item_id")
    preferences = parent["preferences"]
    scored = []
    for item_id in parent["candidate_ids"]:
        item = catalog.loc[item_id]
        score = 0.001 * float(item["popularity"])
        if scenario["dataset"] == "movies":
            score += 2.0 * (preferences.get("genre") == item["genre"])
            score += 1.2 * (preferences.get("tone") == item["tone"])
            score += 0.6 * (preferences.get("pace") == item["pace"])
            if preferences.get("avoid_horror") and bool(item["horror"]):
                score -= 4.0
        else:
            score += 2.0 * (preferences.get("cuisine") == item["cuisine"])
            score += 1.2 * (preferences.get("ambience") == item["ambience"])
            if preferences.get("vegan_required") and not bool(item["vegan"]):
                score -= 4.0
        score += rng.normal(0, STOCHASTICITY)
        scored.append((item_id, float(score)))
    scored.sort(key=lambda pair: pair[1], reverse=True)

    if scenario["planted_stage"] == "Rank" and descriptor == scenario["planted_target"] and len(scored) > 1:
        key, value = scenario["bias_fact_key"], scenario["bias_fact_value"]
        matching = [p for p in scored if key in catalog.columns and catalog.loc[p[0], key] == value]
        chosen = matching[-1] if matching else scored[-1]
        scored = [chosen] + [p for p in scored if p[0] != chosen[0]]

    ranked = [item for item, _ in scored[:TOP_K]]
    rationales = []
    for item_id in ranked:
        item = catalog.loc[item_id]
        keys = ["genre", "tone", "pace"] if scenario["dataset"] == "movies" else ["cuisine", "ambience", "price"]
        tags = [f"{key} match" for key in keys if preferences.get(key) == item[key]]
        rationales.append(tags or ["popularity"])
    return {"stage": "Rank", "ranked_ids": ranked, "scores": dict(scored), "rationales": rationales}


def explain_stage(scenario, parent, descriptor, seed):
    rng = np.random.default_rng(seed)
    ranked = parent["ranked_ids"]
    top_item = ranked[0] if ranked else None
    catalog = CATALOGS[scenario["dataset"]].set_index("item_id")
    reasons = []
    if top_item is not None:
        item = catalog.loc[top_item]
        preferences = parent["preferences"]
        keys = ["genre", "tone", "pace"] if scenario["dataset"] == "movies" else ["cuisine", "ambience", "price"]
        reasons = [fact_str(k, preferences[k]) for k in keys if k in preferences and preferences[k] == item[k]]

    if scenario["planted_stage"] == "Explain" and descriptor == scenario["planted_target"]:
        reasons.append(fact_str(scenario["bias_fact_key"], scenario["bias_fact_value"]))

    reasons = sorted(set(reasons or ["popular=true"]))
    if random_event(rng, STOCHASTICITY / 2):
        reasons = reasons[:1]
    if scenario.get("validity_fault") == "Explain" and descriptor == scenario["planted_target"]:
        alternatives = [i for i in catalog.index if i != top_item]
        top_item = alternatives[0]
    title = catalog.loc[top_item, "title"] if top_item is not None else "None"
    return {"stage": "Explain", "top_item": top_item, "reason_tags": reasons,
            "text": f"I recommend {title} because of {', '.join(reasons)}."}


def memory_stage(scenario, parent, descriptor, seed):
    rng = np.random.default_rng(seed)
    facts = [{"fact": f, "source": "user_or_elicit"} for f in parent["preference_facts"]]
    if scenario["planted_stage"] == "Memory" and descriptor == scenario["planted_target"]:
        facts.append({"fact": fact_str(scenario["bias_fact_key"], scenario["bias_fact_value"]),
                      "source": "inferred_from_descriptor"})
    for reason in parent.get("reason_tags", []):
        if reason.startswith(scenario["bias_fact_key"] + "=") and reason not in [x["fact"] for x in facts]:
            facts.append({"fact": reason, "source": "explanation"})
    if facts and random_event(rng, STOCHASTICITY / 2):
        facts = facts[:-1]
    return {"stage": "Memory", "facts": facts}


def run_stage(stage, scenario, parent, descriptor, seed):
    return {"Elicit": elicit_stage, "Retrieve": retrieve_stage, "Rank": rank_stage,
            "Explain": explain_stage, "Memory": memory_stage}[stage](scenario, parent, descriptor, seed)


def stage_parent(stage, scenario, trajectory=None):
    if stage == "Elicit":
        return {"message": scenario["message"],
                "revealed_preferences": scenario["revealed_preferences"],
                "unresolved_facets": scenario["unresolved_facets"],
                "history": [], "prior_memory": []}
    if stage == "Retrieve":
        return {"preferences": facts_to_dict(trajectory["Elicit"]["preference_facts"]),
                "history": [scenario["message"], trajectory["Elicit"].get("question_text")], "memory": []}
    if stage == "Rank":
        return {"preferences": facts_to_dict(trajectory["Elicit"]["preference_facts"]),
                "candidate_ids": trajectory["Retrieve"]["candidate_ids"]}
    if stage == "Explain":
        return {"preferences": facts_to_dict(trajectory["Elicit"]["preference_facts"]),
                "ranked_ids": trajectory["Rank"]["ranked_ids"],
                "rationales": trajectory["Rank"]["rationales"]}
    if stage == "Memory":
        return {"preference_facts": trajectory["Elicit"]["preference_facts"],
                "reason_tags": trajectory["Explain"]["reason_tags"],
                "ranked_ids": trajectory["Rank"]["ranked_ids"]}
    raise ValueError(stage)


def run_trajectory(scenario, descriptor, repeat):
    trajectory = {}
    for stage in STAGES:
        parent = stage_parent(stage, scenario, trajectory)
        trajectory[stage] = run_stage(stage, scenario, parent, descriptor,
                                      stage_seed(scenario["scenario_id"], repeat, stage))
    return trajectory


def stage_metrics(stage, left, right):
    if stage == "Elicit":
        return {"ask_gap": float(left["ask"] != right["ask"]),
                "target_gap": float(left.get("question_target") != right.get("question_target")),
                "pref_jaccard": jaccard_distance(left["preference_facts"], right["preference_facts"])}
    if stage == "Retrieve":
        return {"query_jaccard": jaccard_distance(left["query_terms"], right["query_terms"]),
                "candidate_jaccard": jaccard_distance(left["candidate_ids"][:TOP_K], right["candidate_ids"][:TOP_K])}
    if stage == "Rank":
        return {"top1_gap": float((left["ranked_ids"] or [None])[0] != (right["ranked_ids"] or [None])[0]),
                "rbo_distance": 1.0 - rbo_score(left["ranked_ids"], right["ranked_ids"])}
    if stage == "Explain":
        return {"top_item_gap": float(left["top_item"] != right["top_item"]),
                "reason_jaccard": jaccard_distance(left["reason_tags"], right["reason_tags"])}
    if stage == "Memory":
        return {"fact_jaccard": jaccard_distance([r["fact"] for r in left["facts"]],
                                                 [r["fact"] for r in right["facts"]])}
    raise ValueError(stage)


def primary_distance(stage, left, right):
    return stage_metrics(stage, left, right)[PRIMARY_METRIC[stage]]

print("Agent, parents, and reference-free distances ready.")

## 7. Proposal 1 — the per-stage reference

For each stage, `stage_reference` returns the set the stage *should* have delivered, taken
from the controlled oracle, and `delivered` returns the part of the actual output that lands
inside it. Two numbers follow:

| quantity | meaning |
|---|---|
| `benefit` | \(\lvert\text{delivered}\rvert / \lvert\mathcal{G}_s\rvert\) — how much of the reference the stage delivered, in \([0,1]\). Rank uses nDCG@K instead, because order matters there. |
| `sim_pref` | similarity between the delivered set under identity \(a\) and under neutral — CFaiRLLM's \(\beta_{pref}\), lifted to stage level |

The reference by stage:

- **Elicit** — the facets genuinely still unresolved, plus the facts that are actually true.
  A question about an already-settled facet delivers nothing.
- **Retrieve** — the oracle's relevant items.
- **Rank** — the oracle's relevance scores, scored with nDCG@K.
- **Explain** — the claims that are actually supported by the user's preferences and the
  recommended item. A stereotyped reason is not in the reference, so it cannot earn benefit.
- **Memory** — the facts that are actually true of the user. A fact inferred from the
  descriptor is not in the reference.

In [ ]:
#@title 7A. Stage reference sets and delivered mass
def stage_reference(stage, scenario, parent):
    """The set the stage ought to deliver, from the controlled oracle only."""
    truth = scenario["true_preferences"]
    if stage == "Elicit":
        useful_questions = set(scenario["unresolved_facets"])
        true_facts = set(dict_to_facts(truth))
        return {"questions": useful_questions, "facts": true_facts}
    if stage == "Retrieve":
        relevant, _ = relevant_items(scenario, k=10)
        return {"items": set(relevant)}
    if stage == "Rank":
        _, relevance = relevant_items(scenario, k=30)
        return {"relevance": relevance}
    if stage == "Explain":
        supported = set(dict_to_facts(truth))
        catalog = CATALOGS[scenario["dataset"]].set_index("item_id")
        keys = ["genre", "tone", "pace"] if scenario["dataset"] == "movies" else ["cuisine", "ambience", "price"]
        top = parent.get("ranked_ids", [None])[0] if parent.get("ranked_ids") else None
        if top is not None and top in catalog.index:
            supported |= {fact_str(k, catalog.loc[top, k]) for k in keys}
        return {"claims": supported}
    if stage == "Memory":
        return {"facts": set(dict_to_facts(truth))}
    raise ValueError(stage)


def delivered(stage, output, reference):
    """The part of the stage output that lands inside the reference set."""
    if stage == "Elicit":
        asked = {output["question_target"]} if output.get("ask") and output.get("question_target") else set()
        good_question = asked & reference["questions"]
        good_facts = set(output["preference_facts"]) & reference["facts"]
        return good_question | good_facts
    if stage == "Retrieve":
        return set(output["candidate_ids"][:TOP_K]) & reference["items"]
    if stage == "Rank":
        return [i for i in output["ranked_ids"] if reference["relevance"].get(i, -1) >= 0]
    if stage == "Explain":
        return set(output["reason_tags"]) & reference["claims"]
    if stage == "Memory":
        return {r["fact"] for r in output["facts"]} & reference["facts"]
    raise ValueError(stage)


def benefit_level(stage, output, reference):
    """Absolute usefulness in [0, 1]: how much of the reference the stage delivered."""
    if stage == "Rank":
        return ndcg_at_k(output["ranked_ids"], reference["relevance"], TOP_K)
    if stage == "Elicit":
        size = len(reference["questions"]) + len(reference["facts"])
        return len(delivered(stage, output, reference)) / size if size else np.nan
    if stage == "Retrieve":
        size = len(reference["items"])
        return len(delivered(stage, output, reference)) / size if size else np.nan
    if stage == "Explain":
        tags = set(output["reason_tags"])
        return len(tags & reference["claims"]) / len(tags) if tags else np.nan
    if stage == "Memory":
        facts = {r["fact"] for r in output["facts"]}
        return len(facts & reference["facts"]) / len(facts) if facts else np.nan
    raise ValueError(stage)


def delivered_similarity(stage, out_a, out_neutral, ref_a, ref_neutral):
    """CFaiRLLM's beta_pref at stage level: compare only the parts that landed."""
    left, right = delivered(stage, out_a, ref_a), delivered(stage, out_neutral, ref_neutral)
    if stage == "Rank":
        return rbo_score(left, right)
    return jaccard_similarity(left, right)

print("Per-stage reference machinery ready.")

## 8. The audit

Three blocks, run in this order so the separation is structural, not just a convention.

1. **Sensitivity.** The SCAFF crossover, using the scenario's own attribute family. No oracle.
2. **Similarity to neutral.** One run per attribute value plus one neutral run.
3. **Consequence.** Reference sets are built and benefit is scored — the first point at which
   `true_preferences` is read.

In [ ]:
#@title 8A. Audit engine
def crossover_cells(scenario, repeat, value_a, value_b):
    """The four SCAFF cells at every stage, for one pair of attribute values."""
    traj_a = run_trajectory(scenario, value_a, repeat)
    traj_b = run_trajectory(scenario, value_b, repeat)
    rows = []
    for stage in STAGES:
        parent_a = stage_parent(stage, scenario, traj_a)
        parent_b = stage_parent(stage, scenario, traj_b)
        seed = stage_seed(scenario["scenario_id"], repeat, stage)
        y_aa, y_bb = traj_a[stage], traj_b[stage]
        y_ab = run_stage(stage, scenario, parent_a, value_b, seed)   # parent from a, attribute b
        y_ba = run_stage(stage, scenario, parent_b, value_a, seed)   # parent from b, attribute a
        d = lambda l, r: primary_distance(stage, l, r)
        rows.append({
            "scenario_id": scenario["scenario_id"], "dataset": scenario["dataset"],
            "attribute": scenario["attribute"], "planted_stage": scenario["planted_stage"],
            "repeat": repeat, "stage": stage,
            "natural": d(y_aa, y_bb),
            "direct": 0.5 * (d(y_aa, y_ab) + d(y_ba, y_bb)),
            "inherited": 0.5 * (d(y_aa, y_ba) + d(y_ab, y_bb)),
        })
    return rows


def audit():
    sensitivity, similarity, consequence = [], [], []

    for _, row in SCENARIOS.iterrows():
        scenario = row.to_dict()
        family = scenario["attribute"]
        values = ATTRIBUTES[family]
        target = scenario["planted_target"]
        other = next(v for v in values if v != target)

        for repeat in range(N_REPEATS):
            # ---- block 1: sensitivity, reference-free -------------------------------
            sensitivity.extend(crossover_cells(scenario, repeat, target, other))

            # ---- block 2: similarity to a neutral run -------------------------------
            traj_neutral = run_trajectory(scenario, NEUTRAL, repeat)
            trajs = {a: run_trajectory(scenario, a, repeat) for a in values}

            for a in values:
                for stage in STAGES:
                    out_a, out_n = trajs[a][stage], traj_neutral[stage]
                    parent_a = stage_parent(stage, scenario, trajs[a])
                    parent_n = stage_parent(stage, scenario, traj_neutral)
                    ref_a = stage_reference(stage, scenario, parent_a)
                    ref_n = stage_reference(stage, scenario, parent_n)

                    similarity.append({
                        "scenario_id": scenario["scenario_id"], "dataset": scenario["dataset"],
                        "attribute": family, "value": a, "planted_stage": scenario["planted_stage"],
                        "repeat": repeat, "stage": stage,
                        "sim_item": 1.0 - primary_distance(stage, out_a, out_n),
                        "sim_pref": delivered_similarity(stage, out_a, out_n, ref_a, ref_n),
                    })

                    # ---- block 3: consequence -------------------------------------
                    b_a = benefit_level(stage, out_a, ref_a)
                    b_n = benefit_level(stage, out_n, ref_n)
                    consequence.append({
                        "scenario_id": scenario["scenario_id"], "dataset": scenario["dataset"],
                        "attribute": family, "value": a, "planted_stage": scenario["planted_stage"],
                        "repeat": repeat, "stage": stage,
                        "benefit": b_a, "benefit_neutral": b_n,
                        "benefit_delta": (b_a - b_n) if (b_a == b_a and b_n == b_n) else np.nan,
                    })

    return {"sensitivity": pd.DataFrame(sensitivity),
            "similarity": pd.DataFrame(similarity),
            "consequence": pd.DataFrame(consequence)}

print("Audit engine ready.")

In [ ]:
#@title 8B. Execute
RESULTS = audit()
for name, frame in RESULTS.items():
    print(f"{name:12s}: {frame.shape}")
print("\nAudit complete.")

## 9. Proposal 2 — SNSR and SNSV per stage

Averaged over scenarios and repeats first, then dispersed across the values of the attribute —
the FaiRLLM order, not the reverse.

In [ ]:
#@title 9. Dispersion per stage
def dispersion(frame, column):
    """Mean over users/repeats first, then range and population std across attribute values."""
    per_value = (frame.groupby(["attribute", "stage", "value"])[column]
                 .mean().reset_index(name="sim_bar"))
    rows = []
    for (attribute, stage), group in per_value.groupby(["attribute", "stage"]):
        values = group["sim_bar"].tolist()
        rows.append({
            "attribute": attribute, "stage": stage, "coordinate": column,
            "n_values": len(values),
            "min": min(values), "max": max(values),
            "SNSR": max(values) - min(values),
            "SNSV": population_std(values),
        })
    return pd.DataFrame(rows), per_value


sim = RESULTS["similarity"]
disp_item, bar_item = dispersion(sim, "sim_item")
disp_pref, bar_pref = dispersion(sim, "sim_pref")
DISPERSION = pd.concat([disp_item, disp_pref], ignore_index=True)

stage_order = {s: i for i, s in enumerate(STAGES)}
DISPERSION = DISPERSION.sort_values(["coordinate", "attribute", "stage"],
                                    key=lambda c: c.map(stage_order) if c.name == "stage" else c)
print("SNSR / SNSV per stage, both benefit levels:")
display(DISPERSION.round(4))

## 10. The binary identity, checked numerically

For a two-valued attribute the population standard deviation of the two means is exactly half
their range, so SNSV carries no information SNSR does not. The check below asserts it on `sex`
and shows that `age`, with three values, departs from it — which is the whole reason to carry a
ternary attribute in the design.

In [ ]:
#@title 10. SNSV = SNSR / 2 for binary attributes
check = DISPERSION.copy()
check["SNSR_over_2"] = check["SNSR"] / 2
check["identity_holds"] = np.isclose(check["SNSV"], check["SNSR_over_2"], atol=1e-12)

binary = check[check["attribute"] == "sex"]
ternary = check[check["attribute"] == "age"]

print("sex (2 values) — identity should hold everywhere:")
display(binary[["stage", "coordinate", "SNSR", "SNSV", "SNSR_over_2", "identity_holds"]].round(6))
assert binary["identity_holds"].all(), "SNSV != SNSR/2 on a binary attribute — check the estimator"

print("\nage (3 values) — SNSV is now an independent number:")
display(ternary[["stage", "coordinate", "SNSR", "SNSV", "SNSR_over_2", "identity_holds"]].round(6))
print("\nRows where the identity breaks (expected: all of the age rows):",
      int((~ternary["identity_holds"]).sum()), "of", len(ternary))

## 11. Where the reference sits: beside \(D_s^A\), not inside it

The table below is the argument made to Yashar, in numbers. For each stage:

- \(D_s^A\) is the reference-free direct sensitivity — computed in block 1, before any oracle existed.
- `benefit_delta` is the consequence — how much better or worse off the user was, relative to the
  neutral run, once the oracle is consulted.

Two cases the pair can distinguish, which neither coordinate can distinguish alone:
**sensitive but harmless** (high \(D_s^A\), `benefit_delta` near zero — the output changed, the
user lost nothing) and **harmful** (`benefit_delta` clearly negative).

Read the printout rather than assuming: on this toy run every stage that clears the
\(D_s^A\) margin also shows a negative `benefit_delta`, so the harmless case does **not** occur
here. Elicit comes closest to it — `benefit_delta` is slightly positive — but its \(D_s^A\)
sits below the margin, so it is never flagged. Producing a genuine sensitive-but-harmless
instance needs a planted fault that reorders without dropping relevant items; that is worth
adding before this argument is made in the paper.

In [ ]:
#@title 11. Sensitivity next to consequence
sens = RESULTS["sensitivity"]
sens_primary = sens.groupby(["attribute", "stage"])[["natural", "direct", "inherited"]].mean()

cons = RESULTS["consequence"]
cons_by_stage = cons.groupby(["attribute", "stage"])[["benefit", "benefit_neutral", "benefit_delta"]].mean()

side_by_side = sens_primary.join(cons_by_stage).reset_index()
side_by_side = side_by_side.sort_values(["attribute", "stage"],
                                        key=lambda c: c.map(stage_order) if c.name == "stage" else c)
side_by_side = side_by_side.rename(columns={"direct": "D_A", "inherited": "D_Z", "natural": "D_nat"})
print("Reference-free sensitivity (left) and reference-dependent consequence (right):")
display(side_by_side.round(4))

flagged = side_by_side[side_by_side["D_A"] > DIRECT_EFFECT_MARGIN]
print(f"\nStages above the D_A margin of {DIRECT_EFFECT_MARGIN}: {len(flagged)}")
if len(flagged):
    harmless = flagged[flagged["benefit_delta"].abs() < 0.02]
    print(f"  of which the user was no worse off (|benefit_delta| < 0.02): {len(harmless)}")
    display(flagged[["attribute", "stage", "D_A", "benefit_delta"]].round(4))

In [ ]:
#@title 12. Figures
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, coord in zip(axes, ["sim_item", "sim_pref"]):
    sub = DISPERSION[DISPERSION["coordinate"] == coord]
    width = 0.35
    xs = np.arange(len(STAGES))
    for offset, attribute in zip([-width / 2, width / 2], ["sex", "age"]):
        vals = [sub[(sub["attribute"] == attribute) & (sub["stage"] == s)]["SNSR"].mean() for s in STAGES]
        ax.bar(xs + offset, vals, width, label=f"{attribute} ({len(ATTRIBUTES[attribute])} values)")
    ax.set_xticks(xs, STAGES, rotation=20)
    ax.set_title(f"SNSR by stage — {coord}")
    ax.set_ylabel("SNSR (lower = more equal)")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT / "snsr_by_stage.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.axhline(0, color="#888", lw=0.8)
ax.axvline(DIRECT_EFFECT_MARGIN, color="#C24A32", lw=1.0, ls="--", label=f"D_A margin {DIRECT_EFFECT_MARGIN}")
for attribute, marker in [("sex", "o"), ("age", "s")]:
    sub = side_by_side[side_by_side["attribute"] == attribute]
    ax.scatter(sub["D_A"], sub["benefit_delta"], marker=marker, s=70, label=attribute)
    for _, r in sub.iterrows():
        ax.annotate(r["stage"], (r["D_A"], r["benefit_delta"]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("$D_s^A$  (sensitivity, reference-free)")
ax.set_ylabel("benefit delta vs neutral  (consequence)")
ax.set_title("Sensitive is not the same as harmful")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "sensitivity_vs_consequence.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
#@title 13. Export
for name, frame in RESULTS.items():
    frame.to_csv(OUT / f"{name}.csv", index=False)
DISPERSION.to_csv(OUT / "dispersion_snsr_snsv.csv", index=False)
side_by_side.to_csv(OUT / "sensitivity_vs_consequence.csv", index=False)

manifest = {
    "seed": SEED,
    "datasets": DATASETS,
    "attributes": ATTRIBUTES,
    "neutral_condition": NEUTRAL,
    "scenarios_per_dataset": N_SCENARIOS_PER_DATASET,
    "repeats": N_REPEATS,
    "top_k": TOP_K,
    "direct_effect_margin": DIRECT_EFFECT_MARGIN,
    "binary_identity_checked": True,
}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Exported:")
for p in sorted(OUT.iterdir()):
    print(" -", p)

## 14. What this does and does not establish

**Established here.**

- A reference set can be defined per stage from the same controlled oracle that already
  backs the quality metrics, and it slots into the consequence layer without touching
  \(D_s^A\). The ordering of the three blocks in section 8 enforces that.
- SNSR and SNSV can be computed per stage against a neutral run, at both benefit levels.
- \(\mathrm{SNSV} = \mathrm{SNSR}/2\) holds exactly for the binary attribute (asserted in
  section 10) and fails for the three-valued one, so the second statistic only earns its
  place from three values up.
- Memory's `sim_pref` comes out at exactly 1.0 for every condition, so its SNSR is 0 while its
  `sim_item` SNSR is not. This is the reference behaving as intended rather than a defect: the
  planted fact (`tone=romantic`) is not true of the user, so it never enters the reference set,
  and once the output is filtered to what is actually true every condition delivers the same
  facts. The stage is demonstrably sensitive and demonstrably not beneficial — which is
  precisely the distinction the two coordinates exist to draw.

**Not established.**

- No noise floor yet. Every number here should be read against the spread of repeated runs
  under an identical condition; on the toy backend that spread is near zero, but on a served
  model it is not, and SNSR in particular inflates with the number of attribute values under
  the null. That is the next thing to add.
- The oracle is exact by construction. A real reference built from held-out user behaviour,
  as in CFaiRLLM, is noisier and may be missing for some stages.
- Benefit at Explain and Memory is a precision, so it is undefined when a stage emits nothing;
  those cells are NaN rather than zero, and are dropped from the means.